In [43]:
import pandas as pd
import numpy as np
import time
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV

import matplotlib.pyplot as plt
import seaborn as sns
import yaml

with open("../config.yaml", "r") as file:
        config = yaml.safe_load(file)

output_dir = "../data/clean/"
config

{'input_data': {'file1': '../data/raw/2024-10-01_performance_mobile_tiles.parquet',
  'file2': '../data/raw/ADE_4-0_GPKG_WGS84G_FRA-ED2026-01-19.gpkg',
  'file3': '../data/raw/fichier_diffusion_2024.xlsx',
  'file4': '../data/raw/2024-10-01_performance_fixed_tiles.parquet'},
 'output_data': {'file1': '../data/clean/perf_admin_fr_df.csv',
  'file2': '../data/clean/fixed_internet_perf_clean.csv',
  'file3': '../data/clean/performance_table',
  'file4': '../data/clean/pregion_table',
  'file5': '../data/clean/department_table',
  'file6': '../data/clean/commune_table'}}

In [10]:
df = pd.read_csv(config["output_data"]["file2"], dtype= {
                                                        'code_insee_dept': str,
                                                        'code_insee_comm': str,
                                                        'zip_code': str}
                )
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 229007 entries, 0 to 229006
Data columns (total 26 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   avg_down_mbps          229007 non-null  float64
 1   avg_up_mbps            229007 non-null  float64
 2   avg_lat_ms             229007 non-null  int64  
 3   avg_lat_down_ms        229007 non-null  float64
 4   avg_lat_up_ms          229007 non-null  float64
 5   nbr_tests              229007 non-null  int64  
 6   nbr_devices            229007 non-null  int64  
 7   region_name            229007 non-null  str    
 8   code_insee_region      229007 non-null  int64  
 9   code_siren_region      229007 non-null  int64  
 10  department_name        229007 non-null  str    
 11  code_insee_dept        229007 non-null  str    
 12  commune_name           229007 non-null  str    
 13  commune_status         229007 non-null  str    
 14  code_insee_comm        229007 non-null  str    

In [34]:
#Prediction of the Broadband score:
#define the features and target columns
features = df[["avg_down_mbps",
               "avg_up_mbps",
               "avg_lat_ms",
               "avg_lat_down_ms",
               "avg_lat_up_ms",
               "nbr_tests",
               "nbr_devices"]]

target = df[["broadband_score"]]

#Split dataset
X_train, X_test, y_train, y_test= train_test_split(features, target, test_size = 0.2, random_state = 0)

# Data scaling
std_scaler = StandardScaler()

#fit scaler
std_scaler.fit(X_train)
X_train_np = std_scaler.transform(X_train)
X_test_np = std_scaler.transform(X_test)

#Convert np arrays to df
X_train_df = pd.DataFrame(X_train_np, columns = X_train.columns, index = X_train.index)
X_test_df = pd.DataFrame(X_test_np, columns = X_test.columns, index = X_test.index)

#Initialize predictors
knn = KNeighborsRegressor(n_neighbors=10)
lin_reg = LinearRegression()

#Train models
knn.fit(X_train_df, y_train)
lin_reg.fit(X_train, y_train)

#Evaluate models
y_train_predict_knn = knn.predict(X_train_df)
y_test_predict_knn = knn.predict(X_test_df)

y_train_predict_lin_reg = lin_reg.predict(X_train)
y_test_predict_lin_reg = lin_reg.predict(X_test)

print(f"KNN R2 is: {knn.score(X_test_df, y_test): .2f}")
print(f"Linear Regression R2 is:  {lin_reg.score(X_test, y_test): .2f}")

print(f"KNN MAE, {mean_absolute_error(y_test_predict_knn, y_test): .2f}")
print(f"KNN RMSE, {mean_squared_error(y_test_predict_knn, y_test): .2f}")
#print(f"R2 score, {tree.score(X_test_norm_df, y_test): .2f}")

KNN R2 is:  1.00
Linear Regression R2 is:   1.00
KNN MAE,  0.11
KNN RMSE,  0.04


In [42]:
#Checking the relationship between features
df_num = df[["avg_down_mbps",
               "avg_up_mbps",
               "avg_lat_ms",
               "avg_lat_down_ms",
               "nbr_tests",
               "nbr_devices",
               "avg_lat_up_ms"
            ]]
corr = df_num.corr()
display(corr)
print("Correlation of avg_lat_up_ms with the remaining of the num df is:\n", corr['avg_lat_up_ms'].sort_values(ascending = False))

,avg_down_mbps,avg_up_mbps,avg_lat_ms,avg_lat_down_ms,nbr_tests,nbr_devices,avg_lat_up_ms
avg_down_mbps,1.000000,0.861965,-0.168364,-0.252142,0.061557,0.058985,-0.341481
avg_up_mbps,0.861965,1.000000,-0.199754,-0.220909,0.055217,0.053056,-0.352835
avg_lat_ms,-0.168364,-0.199754,1.000000,0.324305,-0.041087,-0.051435,0.181721
avg_lat_down_ms,-0.252142,-0.220909,0.324305,1.000000,-0.030722,-0.031540,0.309639
nbr_tests,0.061557,0.055217,-0.041087,-0.030722,1.000000,0.653378,-0.030948
nbr_devices,0.058985,0.053056,-0.051435,-0.031540,0.653378,1.000000,-0.038712
avg_lat_up_ms,-0.341481,-0.352835,0.181721,0.309639,-0.030948,-0.038712,1.000000


Correlation of avg_lat_up_ms with the remaining of the num df is:
 avg_lat_up_ms      1.000000
avg_lat_down_ms    0.309639
avg_lat_ms         0.181721
nbr_tests         -0.030948
nbr_devices       -0.038712
avg_down_mbps     -0.341481
avg_up_mbps       -0.352835
Name: avg_lat_up_ms, dtype: float64


In [39]:
# Use KNN to predict the average upstream latency: avg_lat_up_ms
X = df[["avg_down_mbps",
               "avg_up_mbps",
               "avg_lat_ms",
               "avg_lat_down_ms",
               "nbr_tests",
               "nbr_devices"]]

y = df[["avg_lat_up_ms"]]

#train, test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

# Scaling the features usning std normalizer
#init scaler
scaler = StandardScaler()

#fit scaler
scaler.fit(X_train)

X_train_np = scaler.transform(X_train)
X_test_np = scaler.transform(X_test)

#initialize predictor
knn = KNeighborsRegressor(n_neighbors=10)

#train model
knn.fit(X_train_np, y_train)

#Evaluate model:
y_predict = knn.predict(X_test_np)

#Print the R2 score
print(f"R2 score using KNN to predict `avg_lat_up_ms` is: {knn.score(X_test_np, y_test): .2f}")



R2 score using KNN to predict `avg_lat_up_ms` is:  0.28


In [44]:
# Hyperparameters tuning Gridsearch
param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "minkowski"],
    "p": [1, 2],
    "leaf_size": [20, 30, 40]
}

#Create KNN class object
knn = KNeighborsRegressor()

grid = GridSearchCV(
    estimator = knn,
    param_grid = param_grid,
    cv=5,
    scoring="r2",
    verbose = 10
)

grid.fit(X_train_np, y_train)

#Print the best hyper parameters:
print("Grid best params: ", grid.best_params_)
print("Grid best R2 score: ", grid.best_score_)


AttributeError: 'GridSearchCV' object has no attribute 'best_param_'

In [46]:
print("Grid best params: ", grid.best_params_)
print("Grid best R2 score: ", grid.best_score_)

Grid best params:  {'leaf_size': 20, 'metric': 'manhattan', 'n_neighbors': 31, 'p': 1, 'weights': 'distance'}
Grid best R2 score:  0.4762706639728541


In [47]:
# Use KNN with the best params to get m=best score : to predict the average upstream latency: avg_lat_up_ms
X = df[["avg_down_mbps",
               "avg_up_mbps",
               "avg_lat_ms",
               "avg_lat_down_ms",
               "nbr_tests",
               "nbr_devices"]]

y = df[["avg_lat_up_ms"]]

#train, test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

# Scaling the features usning std normalizer
#init scaler
scaler = StandardScaler()

#fit scaler
scaler.fit(X_train)

X_train_np = scaler.transform(X_train)
X_test_np = scaler.transform(X_test)

#initialize predictor
#KNeighborsRegressor(**grid.best_params_)
knn = KNeighborsRegressor(n_neighbors=31, weights='distance', p=1, metric = 'manhattan', leaf_size = 20)

#train model
knn.fit(X_train_np, y_train)

#Evaluate model:
y_predict = knn.predict(X_test_np)

#Print the R2 score
print(f"R2 score using KNN to predict `avg_lat_up_ms` is: {knn.score(X_test_np, y_test): .2f}")

R2 score using KNN to predict `avg_lat_up_ms` is:  0.53
